# DATA PREPROCESSING AND FEATURE ENGINEERING IN MACHINE LEARNING

### Objective:

In [1]:
## ● This assignment aims to equip you with practical skills in data preprocessing, feature engineering, and feature selection techniques, 
## .. which are crucial for building efficient machine learning models. 
## ..You will work with a provided dataset to apply various techniques such as scaling, encoding, and feature selection methods including
## ..isolation forest and PPS score analysis.

## ● Dataset: Given "Adult" dataset, which predicts whether income exceeds $50K/yr based on census data.

#### Tasks:

### 1. Data Exploration and Preprocessing:

In [2]:
## •	Load the dataset and conduct basic data exploration (summary statistics, missing values, data types).
## •	Handle missing values as per the best practices (imputation, removal, etc.).
## •	Apply scaling techniques to numerical features:
## •	Standard Scaling
## •	Min-Max Scaling
## •	Discuss the scenarios where each scaling technique is preferred and why.

In [3]:
## Import required libraries for EDA process..
import pandas as pd
import numpy as np

In [4]:
## load and display the dataset..
df = pd.read_csv('adult_with_headers.csv')
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
## inspecting about datatype of each column/feature and presence of null values..
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [6]:
## No null values are there..

In [7]:
df.describe()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [8]:
## One thing that i observed was capital gain and capital loss seems to be highly skewed (by looking at their 3rd quartile and maximum values)..
## Most values are 0.. few very large values seems to be there..
## We will cross check going forward using plots..

## This is where log transformation will comes into picture.. we will handel this using this.. 

In [9]:
## Cross checking for null values..
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

In [10]:
## When i googled about the csv file i saw the mention of Missing values stored as '?' instead of proper NaN..
## Lets cross verify that..

(df == '?').sum()

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

In [11]:
df.isin(['?', ' ?']).sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
dtype: int64

In [12]:
df.shape

(32561, 15)

In [13]:
## The dataset was checked for missing values using both standard null checkes and manual inspection for special charater such as '?', '?'..
## In manual inspection I found out that 'workclass', 'occupation' and 'native_country' has missing values stored as '?' instead of proper NaN..
## I checked for '?'and ' ?' only because these are some common known patterns specially in the data set that we used in this notebook.. also I got a hint regarding this in google.. 

In [14]:
## Now before moving any further to encoding and scalling part we need to deal with these nissing values.. 
## We got 2 options.. either we impute these missing values with mode of the column or we can remove these records completely..

## I think its better to remove them instead of imputing.. 
## The reason behind this decision is that, these missing values are present in categorical columns ('workclass', 'occupation' and 'native_country')..
## Being in those categorical columns, these values do not follow any predictable pattern and also represent a significant portion of the data..
## Imputing such values using mode to such amount of portion (around 1800-2000) could introduce some bais by over representing the most frequent category..
## This will also reduce the comparitive variability in the data.. 
## Also the shape of the dataset that we got is around 32500+ rows, which means it is a huge dataset.. 
## So instead of taking the risk of inducing biasness in the data by imputation, its better to remove these records completely..
## Because after the removal of these missing values we will still have around 30000+ records left in our dataset, which is still more than enough..

In [15]:
## we only checked for '?' and ' ?'.. but there could be more missing values like them with different amount of spaces..
## So to handel this lets remove the extra spaces first, specially for the categorical columns.. 
df = df.apply(lambda x: x.str.strip() if x.dtypes == "ojbect" else x)

In [16]:
## Verify missing values again, to check if spaces are removed succesfully or not..
(df == '?').sum()

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

In [17]:
## I am still not getting the correct counts for missing values..

In [18]:
df['workclass'].unique()

array([' State-gov', ' Self-emp-not-inc', ' Private', ' Federal-gov',
       ' Local-gov', ' ?', ' Self-emp-inc', ' Without-pay',
       ' Never-worked'], dtype=object)

In [19]:
df['workclass'].str.contains(r'\?').sum()

np.int64(1836)

In [20]:
## Now above step clears a lot of things.. 
## (df == ' ?').sum() was showing me the correct missing value counts.. 
## (df == '?').sum() was not showing me the correct count of missing values..
##  I tried removing the spaces using .strip() from categorical column and again checked for (df == '?').sum() but it still failed to show me the correct count..
## This gave me a hint that maybe they contain extra hidden character.. 
## To verify this I checked --> df['workclass'].str.contains(r'\?').sum()..  this time I got the correct output of missing values.. 
## This made sure that missing charaters are not just spaces.. 

In [21]:
df[df['workclass'].str.contains(r'\?', na = False)].head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
27,54,?,180211,Some-college,10,Married-civ-spouse,?,Husband,Asian-Pac-Islander,Male,0,0,60,South,>50K
61,32,?,293936,7th-8th,4,Married-spouse-absent,?,Not-in-family,White,Male,0,0,40,?,<=50K
69,25,?,200681,Some-college,10,Never-married,?,Own-child,White,Male,0,0,40,United-States,<=50K
77,67,?,212759,10th,6,Married-civ-spouse,?,Husband,White,Male,0,0,2,United-States,<=50K
106,17,?,304873,10th,6,Never-married,?,Own-child,White,Female,34095,0,32,United-States,<=50K


In [22]:
## Since .strip() only handles spaces and here it is not working so lets use regex to deal with this..
df['workclass'] = df['workclass'].replace(r'\s*\?\s*', regex = True)
df['occupation'] = df['occupation'].replace(r'\s*\?\s*', regex = True)
df['native_country'] = df['native_country'].replace(r'\s*\?\s*', regex = True)

## The pattern r'\s*\?\s*' was used to match any "?" value with 0 or more spaces before and after it..
## \s* repersents spaces..
## This helps converting all variations like " ?" or " ? " or any possible variation into clean "?" for consistent handeling of missing values..

In [23]:
(df == "?").sum()
## I am still not getting correct counts of missing values in this step even after using regex..

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

In [24]:
## This means root cause is something else.. lets try a final approach..
## Lets replace any value that contain question mark (?) with "?" (without space)..

df['workclass'] = df['workclass'].apply(lambda x: '?' if '?' in str(x) else x)
df['occupation'] = df['occupation'].apply(lambda x: '?' if '?' in str(x) else x)
df['native_country'] = df['native_country'].apply(lambda x: '?' if '?' in str(x) else x)

In [25]:
(df == "?").sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
dtype: int64

In [26]:
## Finally we successfully got the correct count of missing values for -->  (df == "?").sum()
## Any value containing (?) was replaced with "?" to ensure consistent identification of missing values..

In [27]:
## Now as we decided above lets remove missing values instead of imputing..
df = df[(df['workclass'] != '?') & (df['occupation'] != '?') & (df['native_country'] != '?')]

In [28]:
(df == "?").sum()
## Finally all the missing values have been successfully dealt withh.. 
## There are no missing values left in the data..

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

In [29]:
df.shape
## After removing missing values instead of imputaion to avoid bais introduction to the dataset..
## we still got 30000+ records as we assumed which is a great dataset size for analysis..

(30162, 15)

In [30]:
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

In [31]:
## Now although our next step should be scaling according to the assingment order..
## But we should do encoding process before scaling because thats the sequence in standard ML pipeline..

In [32]:
df = df.copy()
## A copy of dataset was created to avoid SettingWithCopyWaring and ensure safe modification of the dataframe..

In [33]:
## Storing numerical columns to a variable..
num_cols = df.select_dtypes(include = 'number' )
num_cols.head()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
0,39,77516,13,2174,0,40
1,50,83311,13,0,0,13
2,38,215646,9,0,0,40
3,53,234721,7,0,0,40
4,28,338409,13,0,0,40


In [34]:
## Storing categorical columns to a variable..
cat_cols = df.select_dtypes(include = 'object' )
cat_cols.head()

,workclass,education,marital_status,occupation,relationship,race,sex,native_country,income
0,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,United-States,<=50K
1,Self-emp-not-inc,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,United-States,<=50K
2,Private,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,United-States,<=50K
3,Private,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,United-States,<=50K
4,Private,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,Cuba,<=50K


### 2. Encoding Techniques:

In [35]:
## •	Apply One-Hot Encoding to categorical variables with less than 5 categories.
## •	Use Label Encoding for categorical variables with more than 5 categories.
## •	Discuss the pros and cons of One-Hot Encoding and Label Encoding.

In [36]:
cat_cols.columns

Index(['workclass', 'education', 'marital_status', 'occupation',
       'relationship', 'race', 'sex', 'native_country', 'income'],
      dtype='object')

In [37]:
## Before proceeding to encoding part we need to decide that first that which categorical columns need one hot encoding..
## and which categorical columns need label encoding..
for i in cat_cols :
    print(i, ":", df[i].nunique())

workclass : 7
education : 16
marital_status : 7
occupation : 14
relationship : 6
race : 5
sex : 2
native_country : 41
income : 2


In [38]:
## Now how do we decide that what type of encoding we need to do on a particular categorical column..?
## (1) We check the presence of order.. if order exists then label encoding.. if no order exixts then one hot encoding..
## (2) Number of categories.. few categories--> one hot encoding.. many categories--> label encoding..
## (3) Tree based models--> prefer label encoding (because they dont assume order strongly)..
## (4) Distance based model--> prefer one hot encoding (because label encoding introduce fake distances)
## (5) Memory and performace.. one hot encoding--> more columns, more memory.. label encoding--> vice versa

## So here i am choosing to decide based on number of categories.. 
## Less than 5 categories--> one hot encoding, more than equal to 5 categories in a column--> label encoding..

In [39]:
## 'sex' and 'income' have less than 5 categories.. (one hot encoding)
## Rest of ther categorical columns has 5 or more than 5 categories.. (label encoding)

#### Label Encoding

In [40]:
from sklearn.preprocessing import LabelEncoder

In [41]:
label_cols = []
for i in cat_cols:
    if i  not in ['sex', 'income']:
        label_cols.append(i)
print(label_cols)

['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'native_country']


In [42]:
le = LabelEncoder()
for i in label_cols:
    df[i] = le.fit_transform(df[i])

#### One-hot Encoding

In [43]:
df = pd.get_dummies(df, columns = ['sex', 'income'], drop_first = True)

In [44]:
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,capital_gain,capital_loss,hours_per_week,native_country,sex_ Male,income_ >50K
0,39,5,77516,9,13,4,0,1,4,2174,0,40,38,True,False
1,50,4,83311,9,13,2,3,0,4,0,0,13,38,True,False
2,38,2,215646,11,9,0,5,1,4,0,0,40,38,True,False
3,53,2,234721,1,7,2,5,0,2,0,0,40,38,True,False
4,28,2,338409,9,13,2,9,5,2,0,0,40,4,False,False


In [45]:
df.shape

(30162, 15)

In [46]:
## One-hot Encoding
## Pros - No false ordering between categories.. works well with most of the ML algorithms
## Cons - Increases the number of the columns (high dimensionality)..this can lead to sparce data (most elements are 0)..

## Label Encoding
## Pros - Memory efficient.. suitable for high-categories columns..
## Cons - Introduce artificial order between categories.. can mislead some algorithms..  

In [47]:
## The dataframe is successfully encoded.. 
## Label encoding was used for high-categories columns to avoid unneccessarey increase of dimensions..
## While one-hot encoding was used for low-categories columns to prevent unintentional ordinal relationships..

#### Scaling part

In [48]:
## Although scaling numerical columns  before encoding is technically possible.....
## But encoding is generally performed first to ensure a consistent and structured preprocessing pipeline.....
## Where all features are in numerical form before applying transformation..
## And most Ml pipeline follows (Cleaning-->Encoding-->Scaling-->Model)

In [49]:
## Now lets apply scaling technique..
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [50]:
## Scale only original numerical columns not encoded one..

In [51]:
## Standard scaling.. 
scaler_std = StandardScaler()
df_std_scaler = pd.DataFrame(scaler_std.fit_transform(num_cols), columns = num_cols.columns)
df_std_scaler.head()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
0,0.042796,-1.062722,1.128918,0.146092,-0.218586,-0.077734
1,0.880288,-1.007871,1.128918,-0.147445,-0.218586,-2.331531
2,-0.033340,0.244693,-0.439738,-0.147445,-0.218586,-0.077734
3,1.108695,0.425240,-1.224066,-0.147445,-0.218586,-0.077734
4,-0.794697,1.406658,1.128918,-0.147445,-0.218586,-0.077734


In [52]:
## MinMax scaling.. 
scaler_minmax = MinMaxScaler()
df_mm_scaler = pd.DataFrame(scaler_minmax.fit_transform(num_cols), columns = num_cols.columns)
df_mm_scaler.head()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
0,0.301370,0.043338,0.800000,0.02174,0.0,0.397959
1,0.452055,0.047277,0.800000,0.00000,0.0,0.122449
2,0.287671,0.137244,0.533333,0.00000,0.0,0.397959
3,0.493151,0.150212,0.400000,0.00000,0.0,0.397959
4,0.150685,0.220703,0.800000,0.00000,0.0,0.397959


In [53]:
## Standard Scalling..
## Standard scaling transforms the data in terms of standard deviation and to have a mean = 0 and standard deviation = 1.
## It is preffered when the data follows a normal distribution or
## for algorithms like SVM, Logistic regression or PCA..

## Min-Max Scaling
## Minmax scaling transforms the data into a fixed range between 0 and 1.. 
## It is useful when the distribution is not normal..
## And for algorithms like KNN and Neural Networks where scale matters (distance based algorithms)

## In short, StandarScaler standardize the data while MinMaxScaler normalize it to a fixed range..

### 3. Feature Engineering:

In [54]:
## •	Create at least 2 new features that could be beneficial for the model. Explain the rationale behind your choices.
## •	Apply a transformation (e.g., log transformation) to at least one skewed numerical feature and justify your choice.

In [55]:
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,capital_gain,capital_loss,hours_per_week,native_country,sex_ Male,income_ >50K
0,39,5,77516,9,13,4,0,1,4,2174,0,40,38,True,False
1,50,4,83311,9,13,2,3,0,4,0,0,13,38,True,False
2,38,2,215646,11,9,0,5,1,4,0,0,40,38,True,False
3,53,2,234721,1,7,2,5,0,2,0,0,40,38,True,False
4,28,2,338409,9,13,2,9,5,2,0,0,40,4,False,False


In [56]:
## To create 2 or more new features as asked in the assignmet, we need to use the original dataset not the scaled one..
## It has to be performed on the original dataset to preserve the meaningful relationship between variables before applying scaling..
## So lets store the original data to a new variable df_org..
df_org = pd.read_csv("adult_with_headers.csv")
df_org.head(2)

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K


In [57]:
## lets deal with null/missing values again for this..
df_org.isin(['?', ' ?']).sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
dtype: int64

In [58]:
## Since we know now that what kind of missing values are present in our data.. lets replace them (missing values in the form of ?) first with '?' (without space)..
df_org['workclass'] = df_org['workclass'].apply(lambda x: '?' if '?' in str(x) else x)
df_org['occupation'] = df_org['occupation'].apply(lambda x: '?' if '?' in str(x) else x)
df_org['native_country'] = df_org['native_country'].apply(lambda x: '?' if '?' in str(x) else x)

In [59]:
## Now remove them..
df_org = df_org[(df_org['workclass'] != '?') & (df_org['occupation'] != '?') & (df_org['native_country'] != '?')]

In [60]:
## Now check for missing values again..
df_org.isin(['?', ' ?']).sum()

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

In [61]:
## To avoid SettingWithCopyWarning lets make a copy again..
df_org = df_org.copy()

In [62]:
## Now lets create 2 new feature as we already eliminated missing values from the dataset..

In [63]:
## We have capital gain and capital loss.. so our first new feature will be capital total..
## Capital gain - capital loss = capital total..
## It will represent the net capital..
df_org['capital_total'] = df_org['capital_gain'] - df_org['capital_loss']

In [64]:
df_org['age'].max()

90

In [65]:
## We have 'age' column.. what we can do is we can create bins of age group..
## This is kind of segmentation which will help the model to become less complicated.. less categories = less complicated model..
df_org['age_group'] = pd.cut(df_org['age'], bins = [0, 25, 45, 65, 90], labels = ['Young', 'Adult', 'Senior', 'Old'])

In [66]:
df_org.head()
## New features have been successfully created

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,capital_total,age_group
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,2174,Adult
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,0,Senior
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,0,Adult
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,0,Senior
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,0,Adult


In [67]:
df_org.describe()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week,capital_total
count,30162.000000,3.016200e+04,30162.000000,30162.000000,30162.000000,30162.000000,30162.000000
mean,38.437902,1.897938e+05,10.121312,1092.007858,88.372489,40.931238,1003.635369
std,13.134665,1.056530e+05,2.549995,7406.346497,404.298370,11.979984,7430.372730
min,17.000000,1.376900e+04,1.000000,0.000000,0.000000,1.000000,-4356.000000
25%,28.000000,1.176272e+05,9.000000,0.000000,0.000000,40.000000,0.000000
50%,37.000000,1.784250e+05,10.000000,0.000000,0.000000,40.000000,0.000000
75%,47.000000,2.376285e+05,13.000000,0.000000,0.000000,45.000000,0.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000,99999.000000


In [68]:
## Here we can see that columns like capital_gain, capital_loss and capital_total(because it was generated by capital gain and capital total)  are highly skewed..
## We can this by comparing their min, 25%, 50% and 75 percentile values.. all are 0.. for such columns we have to apply log transformation to them..
## We will apply log transformation to capital gain and capital loss..
df_org['capital_gain_log'] = np.log1p(df_org['capital_gain'])
df_org['capital_loss_log'] = np.log1p(df_org['capital_loss'])

In [69]:
## log transformation is done to reduce the skewness which in result will stabalizes the variance..
## Hence data will become more suitable for ML algorithms.. 
## log1p() handles zero values safely.. log(0) = error.... log1p(0) = 0

In [70]:
## lets check them..
df_org[['capital_gain', 'capital_gain_log', 'capital_loss', 'capital_loss_log']]

,capital_gain,capital_gain_log,capital_loss,capital_loss_log
0,2174,7.684784,0,0.0
1,0,0.000000,0,0.0
2,0,0.000000,0,0.0
3,0,0.000000,0,0.0
4,0,0.000000,0,0.0
...,...,...,...,...
32556,0,0.000000,0,0.0
32557,0,0.000000,0,0.0
32558,0,0.000000,0,0.0
32559,0,0.000000,0,0.0


In [71]:
## Our log transformtaion is correct.. it will improve model performance.. 
## No error was seen, correct and expected transformation and handeled zeros properly.. 
## Since most values are 0 this means feature is highly skewed hence log1p() was the correct choice.. 

## After this we can perform encoding and scaling process.

### 4. Feature Selection:

In [72]:
## •	Use the Isolation Forest algorithm to identify and remove outliers. Discuss how outliers can affect model performance.
## •	Apply the PPS (Predictive Power Score) to find and discuss the relationships between features. Compare its findings with the correlation matrix.

In [73]:
## Isolation forest for outlier detection..
## It returns the anomaly score of each sample using the IsolationForest algorithm

from sklearn.ensemble import IsolationForest

In [74]:
iso = IsolationForest(contamination = 0.05, random_state = 42)
outliers = iso.fit_predict(df.select_dtypes(include = ['int64', 'float64']))

In [75]:
## 1--> normal data
## -1--> outlier
## Isolation forest is used to detect and remove outliers because as we know outliers can manipulate model performace and negatively impact accuracy..
## Outliers affect mean and variance, hence mislead models especially regression and distanced based models..
## One more thing is we have to use scaled data for isolation forest because it is influenced by feature scales.. 
## Model will focus on higher range features compared to smaller one..
## Hence isolation was applied on scaled numerical features to ensure that all values contribute equally to outlier detection..

In [76]:
## Shape of data set before outlier removal..
df.shape

(30162, 15)

In [77]:
outliers

array([1, 1, 1, ..., 1, 1, 1], shape=(30162,))

In [78]:
## Outlier removal..
df = df[outliers == 1]

In [79]:
## shape after outlier removal..
df.shape

(28653, 15)

### PPS Score

In [80]:
## PPS Score--> Predictive power score..
## It measures how well one feature can predict another (even non-liner relationships)..
## Its better than correlation because correlation only detects linear relationships..
## PPS detect liner + non linear relationships.. and another USP of PPS is it works with both categorical and numerical columns/features.. 

In [81]:
## install PPScore, if haven't worked with yet once..
## !pip install ppscore

In [82]:
## Import and calculate pps
import ppscore as pps
pps_matrix = pps.matrix(df)

In [83]:
pps_matrix.head()

,x,y,ppscore,case,is_valid_score,metric,baseline_score,model_score,model
0,age,age,1.0,predict_itself,True,None,0.0000,1.000000,None
1,age,workclass,0.0,regression,True,mean absolute error,0.4374,0.555571,DecisionTreeRegressor()
2,age,fnlwgt,0.0,regression,True,mean absolute error,77990.0808,79608.164001,DecisionTreeRegressor()
3,age,education,0.0,regression,True,mean absolute error,2.5294,2.555097,DecisionTreeRegressor()
4,age,education_num,0.0,regression,True,mean absolute error,1.7616,1.822966,DecisionTreeRegressor()


In [84]:
## pps pivot..
pps_pivot = pps_matrix.pivot(index = 'x', columns = 'y', values = 'ppscore')

In [85]:
pps_pivot

y,age,capital_gain,capital_loss,education,education_num,fnlwgt,hours_per_week,income_ >50K,marital_status,native_country,occupation,race,relationship,sex_ Male,workclass
x,,,,,,,,,,,,,,,
age,1.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.013893,0.142874,0.0,0.000000,0.0,0.000000,0.000000,0.0
capital_gain,0.006132,1.0,0.0,0.0,0.000000,0.000000,0.0,0.278306,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0
capital_loss,0.000000,0.0,1.0,0.0,0.000000,0.000000,0.0,0.103205,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0
education,0.019321,0.0,0.0,1.0,1.000000,0.000000,0.0,0.221695,0.000000,0.0,0.020523,0.0,0.000000,0.000000,0.0
education_num,0.019321,0.0,0.0,1.0,1.000000,0.000000,0.0,0.221695,0.000000,0.0,0.020523,0.0,0.000000,0.000000,0.0
fnlwgt,0.000000,0.0,0.0,0.0,0.000000,1.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.088109,0.0
hours_per_week,0.002766,0.0,0.0,0.0,0.000000,0.000000,1.0,0.005583,0.000000,0.0,0.000000,0.0,0.000000,0.137230,0.0
income_ >50K,0.037458,0.0,0.0,0.0,0.025673,0.000000,0.0,1.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0
marital_status,0.184641,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,1.000000,0.0,0.000000,0.0,0.160079,0.335815,0.0


In [86]:
## All features shows pps = 1 with themselves.. this expected and confirms that the pps calculation is correct..

## Key predictor of target variable (income>50K) are capital_gain--> 0.278 (highest).. education_num--> 0.221..
## education--> 0.103, workclass--> 0.059 and age--> 0.013 (very weak but present).. 

## This tells us that capital_gain is the most influential feature.. people with higher capital gain are more likely to earn > 50K..
## Education and education_num are also important and directly proportional to income>50K feature..
## Workclass has some predictive power.. its like type of employment  affects income..
## Rest of other features have near 0 pps means they are quite weak predictor of income>50K ..

In [87]:
## Relationship between features..
## There is a strong logical connection between relationship --> marital_status (0.29)
## Also a strong connection between occupation --> education_num can be seen..(0.17).. means job type depends on education level..

In [88]:
## Correlation matrix..
corr_matrix = df.corr()

In [89]:
corr_matrix

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,capital_gain,capital_loss,hours_per_week,native_country,sex_ Male,income_ >50K
age,1.000000,0.074985,-0.079397,0.014910,0.063272,-0.306175,-0.010327,-0.263859,0.040862,0.078297,0.033273,0.115674,0.017091,0.095255,0.248789
workclass,0.074985,1.000000,-0.033834,0.022372,0.036290,-0.033978,0.011937,-0.076277,0.066369,0.023573,-0.011811,0.051532,0.017284,0.080735,0.010303
fnlwgt,-0.079397,-0.033834,1.000000,-0.019806,-0.039887,0.031788,-0.002497,0.009946,-0.031328,-0.012138,-0.015916,-0.027209,-0.059879,0.022594,-0.008545
education,0.014910,0.022372,-0.019806,1.000000,0.302398,-0.030949,-0.033795,0.003251,-0.013454,0.025795,0.031123,0.055135,0.031625,-0.041080,0.067146
education_num,0.063272,0.036290,-0.039887,0.302398,1.000000,-0.046054,0.096944,-0.084111,0.027360,0.118413,0.075942,0.150335,0.053718,-0.005797,0.329823
marital_status,-0.306175,-0.033978,0.031788,-0.030949,-0.046054,1.000000,0.024667,0.178682,-0.072752,-0.044047,-0.040709,-0.186822,-0.010959,-0.113808,-0.193622
occupation,-0.010327,0.011937,-0.002497,-0.033795,0.096944,0.024667,1.000000,-0.054735,0.002565,0.011386,-0.000276,0.015797,0.005753,0.062200,0.048413
relationship,-0.263859,-0.076277,0.009946,0.003251,-0.084111,0.178682,-0.054735,1.000000,-0.112809,-0.062329,-0.084241,-0.261425,0.022126,-0.582958,-0.257887
race,0.040862,0.066369,-0.031328,-0.013454,0.027360,-0.072752,0.002565,-0.112809,1.000000,0.028758,0.050659,0.050653,0.064122,0.086834,0.079041
capital_gain,0.078297,0.023573,-0.012138,0.025795,0.118413,-0.044047,0.011386,-0.062329,0.028758,1.000000,-0.026107,0.067185,0.022578,0.049772,0.218136


In [90]:
## PPS vs Correlation..
## The correlation matrix showed mostly weaker relationship between features with few moderate correlations..
## education and education_num (0.302), sex_male and relationship (-0.582), income with education_num (0.329)..
## income with age (0.248), income with hours_per_week (0.227), income with capital gain(0.218)..
## these values indicates some linear dependencies between features and the target variable..

## However the pps analysis provided deeper insights by identifying predective relationships beyond linear patter..
## PPS highlighted capital_gain(0.278) and education_num (0.221) as strong predictor of income..
## Educations and workclass are having moderate influence on income..
## Additionally PPS captured meaningful non-linear relationship such as marital_status and relationship (0.29)..
## Non linear relationships were not clearly reflected in the correlation matrix..

## Hence we can say that correlation measures only linear relationships..
## While pps is more effective in identifying real world predective power including non linear dependencies..
## That makes PPS a better too comparitively for feature selection in the dataset..

In [91]:
## In this assignment various data preprocessing and feature engineering techniques were applied  to prepare the adult dataset for machine learning..
## Missing values were  handeled properly and categorical features were encoded  using both one-hot and label encoding techniques according to the requirements..
## Numerical features were scaled using both standard scalling and Min-max scaler to ensure uniformity in feature ranges..
## New features such as log transformations were created to handle skewness and improve data representation..
## Outliers were detected and removed using isolation forest algorithm enhansing data quality..
## Feature relationships were analysed using both correlation and PPS..
## Correlation identified the linear relationship, while PPS provided insights in more depth by capturing non-linear dependencies and highlighting key predictors like capital gain and education num for income prediction..
## Overall these steps improved the dataset quality and helped in identifying important features..
## They made the data more suitable for building efficient and accurate machine learning models,,

In [92]:
## for i in cat_cols:
##    print(i, ': ', df[i].value_counts() )
##    print('*'*30)

In [93]:
## for i in num_cols:
##    print(i, ': ', df[i].value_counts() )
##    print('*'*30)